# Evaluating LM Outputs using Rubric + LM Judges

Instead of using BLADE's `EntireAnalysisProcessed` code, we will try an evaluation implementation that reads straight from `multirun_analyses.json` and gives the results to an LLM judge.

## Setup

In [1]:
# imports
import json
from stat_genie.blade_pipeline.llms.config import llm
from stat_genie.blade_pipeline.llms.base import TextGenerator
from os.path import join

In [2]:
# define file paths
analysis_subdir_path = "analysis_output"
multirun_filename = "multirun_analyses.json"
# use multirun analyses file to get analysis code paths
multirun_analyses_path = join(analysis_subdir_path, multirun_filename)
with open(multirun_analyses_path, "r") as file:
    multirun_analyses = json.load(file)
num_analyses = multirun_analyses['n']
analysis_code_filenames = [f"llm_analysis_{i}.py" for i in range(num_analyses)]
analysis_code_paths = [join(analysis_subdir_path, filename) for filename in \
    analysis_code_filenames]
# get config details
llm_provider = "openai"
llm_model = "gpt-5-mini"

## Define Helper Functions

In [3]:
def get_feature_transforms(llm_assistant: TextGenerator, transform_code: str,
                           feature_columns: list[str],
                           feature_description: str):
    """
    Given a list of feature columns, check if the columns are transformed in the
    transform code and return the code that performs the transformation.
    
    Args:
        llm_assistant (TextGenerator): The LLM assistant for the evaluation.
        transform_code (str): The code that performs the transformations.
        feature_columns (list[str]): The list of feature columns to check.
        
    Returns:
        dict: A dictionary of feature columns and the code that performs the transformation.
    """
    system_prompt = """You are an AI Data Analysis Assistant who is an expert at \
        performing data cleaning and preprocessing tasks."""
    transform_responses = []
    for feature_column in feature_columns:
        find_transform_prompt = f"""Given the following code:
            <Code>
            {transform_code}
            </Code>
            and the feature column:
            <Feature Column>
            {feature_column}
            </Feature Column>
            with description:
            <Feature Description>
            {feature_description}
            </Feature Description>
            determine if the column is transformed in the code. \
            If it is, return only the corresponding lines of code that perform the transformation. \
            If it is not, return "No transformation code found."
            """
        response = llm_assistant.generate([{"role": "system",
                                            "content": system_prompt},
                                           {"role": "user",
                                            "content": find_transform_prompt}])
        transform_responses.append(response)
    return transform_responses
                

In [4]:
def get_model_information(llm_assistant: TextGenerator, model_code: str):
    """
    Given modeling code, extract relevant information.
    
    Args:
        llm_assistant (TextGenerator): The LLM assistant for the evaluation.
        model_code (str): The code that defines the model.
        
    Returns:
        dict: A dictionary of model information, particularly model class.
    """
    
    system_prompt = """You are an AI Data Analysis Assistant who is an expert at \
        choosing, identifying, and implementing different types of ML models."""
    
    find_model_prompt = f"""Given the following code:
        <Code>
        {model_code}
        </Code>
        extract relevant information about the model. The returned value should be a dictionary with the following keys:
        1. "model_library": The library or framework used (e.g., "sklearn", "statsmodels", "pytorch", "tensorflow").
        2. "model_class": The specific model class or type (e.g., "LinearRegression", "RandomForestClassifier", "LogisticRegression").
        3. "model_parameters": Any parameters or hyperparameters that are set when instantiating or configuring the model.
        4. "model_formula_fitting_code": The code that defines the model formula and/or the code that fits/trains the model.
        
        The values of the dictionary should be strings.
        """
    
    response = llm_assistant.generate([{"role": "system",
                                        "content": system_prompt},
                                       {"role": "user",
                                        "content": find_model_prompt}])
    
    return response.text[0].content
                

## Extract Features **X** Used in Model

In [5]:
# create dict to store features
features = {}

In [6]:
# loop through analyses
for i, analysis_code_path in enumerate(analysis_code_paths):
    
    # create internal dict for analysis features
    features[i] = {}
    
    # get the features from each analysis
    # this should include the independent and control variables
    ind_vars = multirun_analyses['analyses'][str(i)]['cvars']['ivs']
    control_vars = multirun_analyses['analyses'][str(i)]['cvars']['controls']

    # get any lines from the transform code that represent transformations
    # of the independent or control variables
    transform_code = multirun_analyses['analyses'][str(i)]['transform_code']
    # for each variable in ind_vars, check if it is transformed
    # in the transform_code by using an LLM assistant
    llm_assistant = llm(provider=llm_provider, model=llm_model)
    for dict_idx, var in enumerate(ind_vars):
        transform_responses = get_feature_transforms(llm_assistant,
                                                     transform_code,
                                                     var['columns'],
                                                     var['description'])
        ind_vars[dict_idx]['transform_code'] = [response.text[0].content for \
            response in transform_responses]
        
    # save updated independent variables in features dict
    features[i]['independent_variables'] = ind_vars
    
    # tkae same approach for control variables
    for dict_idx, var in enumerate(control_vars):
        transform_responses = get_feature_transforms(llm_assistant,
                                                     transform_code,
                                                     var['columns'],
                                                     var['description'])
        control_vars[dict_idx]['transform_code'] = [response.text[0].content for \
            response in transform_responses]
    
    # save updated control variables in features dict
    features[i]['control_variables'] = control_vars

[2025-11-11 02:23:41.53][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-11-11 02:23:41.85][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.


In [7]:
# view feature dictionary to ensure correctness
features

{0: {'independent_variables': [{'description': 'Name femininity rating (continuous). This is the mean masculinity-femininity rating where larger values indicate more feminine names. Standardized for interpretability and to aid model convergence.',
    'columns': ['masfem_std'],
    'transform_code': ["df['masfem_std'] = (df['masfem'] - df['masfem'].mean()) / (df['masfem'].std(ddof=0) if df['masfem'].std(ddof=0) != 0 else 1.0)"]},
   {'description': 'Binary gender label for the hurricane name (0 = male, 1 = female). Included as an alternative operationalization of name gender.',
    'columns': ['gender_mf'],
    'transform_code': ["# Ensure gender_mf is binary numeric (0/1)\ndf['gender_mf'] = pd.to_numeric(df['gender_mf'], errors='coerce')"]}],
  'control_variables': [{'description': 'Maximum wind speed at landfall (measures storm strength/severity).',
    'is_moderator': False,
    'moderator_on': None,
    'columns': ['wind'],
    'transform_code': ["df['wind'] = pd.to_numeric(df['win

## Extract Response *y* used in Model

In [8]:
# loop through analyses
for i, analysis_code_path in enumerate(analysis_code_paths):
    
    # get the features from each analysis
    # this should include the independent and control variables
    response_vars = multirun_analyses['analyses'][str(i)]['cvars']['dv']

    # get any lines from the transform code that represent transformations
    # of the independent or control variables
    transform_code = multirun_analyses['analyses'][str(i)]['transform_code']
    # for each variable in response_vars, check if it is transformed
    # in the transform_code by using an LLM assistant
    llm_assistant = llm(provider=llm_provider, model=llm_model)
    # for dict_idx, var in enumerate(response_vars):
    transform_responses = get_feature_transforms(llm_assistant,
                                                 transform_code,
                                                 response_vars['columns'],
                                                 response_vars['description'])
    response_vars['transform_code'] = [response.text[0].content \
        for response in transform_responses]

    # save updated response variables in features dict
    features[i]['response_variables'] = response_vars

[2025-11-11 02:23:42.15][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-11-11 02:23:42.39][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.


In [9]:
# view feature dictionary to ensure correctness
features

{0: {'independent_variables': [{'description': 'Name femininity rating (continuous). This is the mean masculinity-femininity rating where larger values indicate more feminine names. Standardized for interpretability and to aid model convergence.',
    'columns': ['masfem_std'],
    'transform_code': ["df['masfem_std'] = (df['masfem'] - df['masfem'].mean()) / (df['masfem'].std(ddof=0) if df['masfem'].std(ddof=0) != 0 else 1.0)"]},
   {'description': 'Binary gender label for the hurricane name (0 = male, 1 = female). Included as an alternative operationalization of name gender.',
    'columns': ['gender_mf'],
    'transform_code': ["# Ensure gender_mf is binary numeric (0/1)\ndf['gender_mf'] = pd.to_numeric(df['gender_mf'], errors='coerce')"]}],
  'control_variables': [{'description': 'Maximum wind speed at landfall (measures storm strength/severity).',
    'is_moderator': False,
    'moderator_on': None,
    'columns': ['wind'],
    'transform_code': ["df['wind'] = pd.to_numeric(df['win

## Extract Model Class Used

In [10]:
# create dict to store model information
model_info = {}

In [11]:
# loop through analyses
for i, analysis_code_path in enumerate(analysis_code_paths):

    # get model code
    model_code = multirun_analyses['analyses'][str(i)]['m_code']

    # use helper function to get model information
    model_info[i] = get_model_information(llm_assistant, model_code)

[2025-11-11 02:23:42.61][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-11 02:23:57.21][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  14.60 seconds
[2025-11-11 02:23:57.21][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)


In [12]:
# view model_info to ensure correctness
model_info

{0: '{\n  "model_library": "statsmodels (imported as statsmodels.api as sm)",\n  "model_class": "sm.GLM with family=sm.families.NegativeBinomial (fallback: sm.GLM with family=sm.families.Poisson)), and sm.OLS for the log-damage robustness check",\n  "model_parameters": "GLM NegativeBinomial: family=sm.families.NegativeBinomial() (uses default log link); fallback GLM Poisson: family=sm.families.Poisson() with fit(cov_type=\'HC0\') for robust SEs; OLS: sm.OLS(...).fit(cov_type=\'HC1\') for robust SEs. Design matrix: predictors = [\'masfem_std\',\'gender_mf\',\'wind\',\'category\',\'min\',\'year\',\'elapsedyrs\',\'masfem_x_category\']; constant added via sm.add_constant(X_model). Interaction created as masfem_x_category = masfem_std * (category - category.mean()).",\n  "model_formula_fitting_code": "try:\\n    nb_model = sm.GLM(y_counts, X_model, family=sm.families.NegativeBinomial()).fit()\\n    results[\'neg_bin_alldeaths\'] = nb_model\\nexcept Exception as e:\\n    pois = sm.GLM(y_coun